# 🚀 Construction PPE YOLOv8 Training on Google Colab (Free T4 GPU)

This notebook trains an optimized YOLOv8 model on the Construction PPE benchmark dataset and downloads `best.pt` directly to your computer.

**Estimated time:** ~8-10 minutes on a free T4 GPU.

In [ ]:
# Step 1: Verify Free T4 GPU & Install Ultralytics
!nvidia-smi
!pip install -q ultralytics

In [ ]:
# Step 2: Upload construction-ppe.ndjson (745 KB from your Downloads)
from google.colab import files
import os

if not os.path.exists('construction-ppe.ndjson'):
    print('Please select construction-ppe.ndjson from your computer:')
    uploaded = files.upload()
else:
    print('construction-ppe.ndjson is already uploaded!')

In [ ]:
# Step 3: Fast Download & YOLO Conversion (~10 seconds on Colab network)
import json, time, os, yaml, urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

PROJECT_CLASS_NAMES = [
    'Hardhat', 'Mask', 'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest',
    'Person', 'Safety Cone', 'Safety Vest', 'machinery', 'vehicle'
]
MAP = {0: 0, 7: 2, 5: 4, 6: 5, 2: 7}
split_map = {'train': 'train', 'val': 'valid', 'test': 'test'}

records = []
with open('construction-ppe.ndjson', 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line.strip())
        if item.get('type') == 'image':
            records.append(item)

base = Path('data_ppe')
download_tasks = []
for item in records:
    split = split_map.get(item.get('split', 'train'), 'train')
    fn = item['file']
    img_dest = base / split / 'images' / fn
    lbl_dest = base / split / 'labels' / f"{Path(fn).stem}.txt"
    img_dest.parent.mkdir(parents=True, exist_ok=True)
    lbl_dest.parent.mkdir(parents=True, exist_ok=True)
    lines = []
    for b in item.get('annotations', {}).get('boxes', []):
        cid = int(b[0])
        if cid in MAP:
            lines.append(f"{MAP[cid]} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}")
    with open(lbl_dest, 'w') as f:
        f.write('\n'.join(lines) + ('\n' if lines else ''))
    download_tasks.append((item['url'], img_dest))

def dl(t):
    u, d = t
    if not d.exists() or d.stat().st_size == 0:
        try:
            req = urllib.request.Request(u, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=15) as r:
                with open(d, 'wb') as fp:
                    fp.write(r.read())
        except Exception:
            pass

print(f'Downloading {len(download_tasks)} images with 32 worker threads...')
with ThreadPoolExecutor(32) as ex:
    list(ex.map(dl, download_tasks))

yaml_cfg = {
    'path': str(base.resolve()),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': len(PROJECT_CLASS_NAMES),
    'names': PROJECT_CLASS_NAMES
}
with open('data_ppe/data.yaml', 'w') as f:
    yaml.dump(yaml_cfg, f, sort_keys=False)
print('✅ Dataset prepared successfully for YOLO!')

In [ ]:
# Step 4: Train YOLOv8 on T4 GPU (~8-10 mins)
from ultralytics import YOLO

model = YOLO('yolov8s.pt')  # YOLOv8 small: optimal balance of speed & mAP
results = model.train(
    data='data_ppe/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project='runs',
    name='ppe_model',
    exist_ok=True
)

In [ ]:
# Step 5: Automatically download best.pt
from google.colab import files
import os

best_weights = 'runs/ppe_model/weights/best.pt'
if os.path.exists(best_weights):
    print('✅ Training complete! Downloading best.pt...')
    files.download(best_weights)
else:
    print('⚠️ Weights not found at:', best_weights)